# Zheng et al. (2023) SVM on the 80 splits

Re-implementation of the NLOS detector of Zheng et al., *Remote Sensing* 15(10), 2676, 2023, with their Rician-K and skewness baselines (`zheng_core.py`). Settings that the paper does not state are collected in `zheng_core.UNSTATED`. Run `prepare_inputs.py` once before this notebook and `make_report.py` afterwards.

In [ ]:
import os
PROTOCOLS = ["lopo", "l2po"]
TABLE3 = True
RUN_BASELINES = True
RUN_SAME_ENV = True
RUN_SENSITIVITY = True
TIMING_ONLY = False
N_JOBS = os.cpu_count()
CACHE_SIZE = 200
MAX_ITER = -1
RESUME_FROM = None

In [ ]:
import glob, json, shutil, sys, time
import numpy as np

ON_KAGGLE = os.path.isdir("/kaggle/input")
if ON_KAGGLE:
    def find(pattern):
        hits = sorted(glob.glob(os.path.join("/kaggle/input", "**", pattern), recursive=True))
        assert hits, f"not found under /kaggle/input: {pattern}"
        return hits[0]
    PV2 = os.path.dirname(os.path.dirname(find("robustness/common.py")))
    MODULE = os.path.dirname(find("zheng_core.py"))
    print("files under /kaggle/input:", sorted(glob.glob("/kaggle/input/*/*"))[:12])
    INPUTS = os.path.join(MODULE, "inputs")
    RESULTS_DIR = "/kaggle/working/zheng_results"
else:
    MODULE = os.path.abspath(".")
    assert os.path.exists(os.path.join(MODULE, "zheng_core.py")), "run this notebook from Zheng2023_SVM"
    PV2 = os.path.dirname(MODULE)
    INPUTS = os.path.join(MODULE, "inputs")
    RESULTS_DIR = os.path.join(MODULE, "results")
COMMON_DIR = os.path.join(PV2, "robustness")
CNN_CORE = os.path.join(PV2, "CNN_LSTM", "cnn_lstm_core.py")
PATHS = (COMMON_DIR, CNN_CORE)
sys.path.insert(0, MODULE)
import zheng_core as zc
zc.setup(*PATHS)
cm = zc.cm
os.makedirs(RESULTS_DIR, exist_ok=True)

if RESUME_FROM is None and ON_KAGGLE:
    old = glob.glob("/kaggle/input/**/zheng_results/summary.json", recursive=True) + \
          glob.glob("/kaggle/input/**/zheng_results/**/splits_*.jsonl", recursive=True)
    RESUME_FROM = (old[0].split("/zheng_results/")[0] + "/zheng_results") if old else None
if RESUME_FROM:
    for src in glob.glob(os.path.join(RESUME_FROM, "**", "*.jsonl"), recursive=True):
        dst = os.path.join(RESULTS_DIR, os.path.relpath(src, RESUME_FROM))
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copyfile(src, dst)
    print("resumed from", RESUME_FROM)
print("PV2       ", PV2)
print("inputs    ", INPUTS)
print("results   ", RESULTS_DIR)
print("CPU cores ", os.cpu_count())

## Step 1: Inputs and Hampel filter

Raw amplitudes of antenna 0 on the 52 data subcarriers, Hampel-filtered and min-max normalised per CSI vector, inside each cluster.

In [ ]:
A, y, ants, IDX = zc.load_inputs(INPUTS, "original")
A_eq, _, ants_eq, _ = zc.load_inputs(INPUTS, "snr_equalised")
POS = ants.index(zc.MAIN_ANTENNA)
assert ants_eq[0] == zc.MAIN_ANTENNA and ants[0] == zc.MAIN_ANTENNA
F, share = zc.filter_clusters(A, POS)
F_eq_raw, share_eq = zc.filter_clusters(A_eq, 0)
F_shift = {c: (F_eq_raw[c] if c in cm.LOS_CLUSTERS else F[c]) for c in cm.ALL_CLUSTERS}
for c in cm.NLOS_CLUSTERS:
    assert np.array_equal(F_eq_raw[c], F[c])
FEAT = zc.svm_features(F)
FEAT_SHIFT = zc.svm_features(F_shift)
METHODS = zc.THRESHOLD_METHODS + zc.DIAGNOSTICS
STAT = {m: {c: zc.frame_statistic(F[c], m) for c in cm.ALL_CLUSTERS} for m in METHODS}
STAT_SHIFT = {m: {c: zc.frame_statistic(F_shift[c], m) for c in cm.ALL_CLUSTERS} for m in METHODS}
assert all(np.allclose(FEAT[c].min(axis=1), 0) and np.allclose(FEAT[c].max(axis=1), 1) for c in cm.ALL_CLUSTERS)
print("settings not stated in the paper:", zc.UNSTATED)

with open(os.path.join(RESULTS_DIR, "hampel.json"), "w") as f:
    json.dump({"settings": zc.UNSTATED["hampel"], "antenna": zc.MAIN_ANTENNA, "replaced_share_original": share,
               "replaced_share_snr_equalised": {c: share_eq[c] for c in cm.LOS_CLUSTERS}}, f, indent=1)
print("share of values replaced by the Hampel filter (antenna 0):")
for c in cm.ALL_CLUSTERS:
    extra = f"   SNR-equalised {share_eq[c] * 100:5.2f} %" if c in cm.LOS_CLUSTERS else ""
    print(f"  {c:14s} {share[c] * 100:5.2f} %{extra}")
print("\nmean frame power (dB, gain-only statistic), per cluster:")
for c in cm.ALL_CLUSTERS:
    print(f"  {c:14s} {STAT['gain_only'][c].mean():7.2f} dB   (SNR-equalised: {STAT_SHIFT['gain_only'][c].mean():7.2f} dB)")

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return {r["name"]: r for r in (json.loads(l) for l in f if l.strip())}

SPLITS = {p: cm.make_splits(p) for p in PROTOCOLS}
for p in PROTOCOLS:
    ref = load_jsonl(os.path.join(PV2, "SVM", "results", f"splits_{p}.jsonl"))
    assert set(ref) == {s["name"] for s in SPLITS[p]}, p
    for s in SPLITS[p]:
        zc.check_split(s)
        n_tr = sum(len(F[c]) for c in s["train"]); n_te = sum(len(F[c]) for c in s["test"])
        assert (n_tr, n_te) == (ref[s["name"]]["n_train"], ref[s["name"]]["n_test"]), s["name"]
        assert sum(len(F[c]) for c in s["test"] if y[c] == 1) == sum(len(F[c]) for c in s["test"] if c.startswith("LOS"))
print("splits and frame counts match the stored SVM results:",
      {p: len(SPLITS[p]) for p in PROTOCOLS})

## Step 2: Timing of one split

In [ ]:
split0 = SPLITS["lopo"][0] if "lopo" in SPLITS else cm.make_splits("lopo")[0]
t0 = time.time()
r0 = zc.run_svm_split(split0, FEAT, y, FEAT_SHIFT, table3=TABLE3, cache_size=CACHE_SIZE, max_iter=MAX_ITER)
t_split = time.time() - t0
print(f"{split0['name']}: {t_split:.0f} s   RBF accuracy {r0['accuracy'] * 100:.2f} %")
if TABLE3:
    print("per kernel: accuracy %, fit time s:",
          {k: (round(v["accuracy"] * 100, 2), round(v["fit_seconds"], 1)) for k, v in r0["table3_kernels"].items()})
n_split_runs = sum(len(SPLITS[p]) for p in PROTOCOLS) + (4 * 20 if RUN_SENSITIVITY else 0)
est = t_split * n_split_runs / max(1, N_JOBS)
print(f"\nestimate for {n_split_runs} split runs with {N_JOBS} parallel jobs: about {est / 3600:.1f} h "
      "(L2PO splits have fewer training frames and 3 inner folds, so they are faster)")
if TIMING_ONLY:
    raise SystemExit("TIMING_ONLY = True: stopping here")

## Step 3: Main run

RBF SVM on all 80 splits, tested per frame and per 50-frame window, and with the SNR-equalised LOS test clusters (condition B). With `TABLE3 = True` the linear, polynomial and sigmoid kernels run as well.

In [ ]:
from joblib import Parallel, delayed

def run_svm_all(protocol, feat, feat_shift, out_dir, splits=None, table3=TABLE3, antenna=zc.MAIN_ANTENNA, tag=""):
    path = os.path.join(out_dir, f"splits_{protocol}.jsonl")
    done = cm.load_done(path)
    todo = [s for s in (splits or SPLITS[protocol]) if s["name"] not in done]
    print(f"SVM {tag}{protocol:12s}: {len(done)} done, {len(todo)} to run")
    if not todo:
        return
    t0 = time.time()
    jobs = (delayed(zc.svm_split_worker)(PATHS, s, feat, y, feat_shift, table3=table3,
                                         cache_size=CACHE_SIZE, max_iter=MAX_ITER) for s in todo)
    for k, r in enumerate(Parallel(n_jobs=N_JOBS, return_as="generator_unordered")(jobs), 1):
        r["antenna"] = antenna
        cm.append_result(path, r)
        b = f"  B {r['test_shift']['frame']['accuracy'] * 100:6.2f} %" if "test_shift" in r else ""
        print(f"  [{k}/{len(todo)}] {r['name']:24s} acc {r['accuracy'] * 100:6.2f} %{b}"
              f"  ({(time.time() - t0) / 60:.1f} min)", flush=True)

for p in PROTOCOLS:
    run_svm_all(p, FEAT, FEAT_SHIFT, os.path.join(RESULTS_DIR, "svm"))

## Step 4: Threshold baselines

Rician K-factor, skewness and frame power, with threshold and direction fitted on the training clusters.

In [ ]:
if RUN_BASELINES:
    for m in METHODS:
        for p in PROTOCOLS:
            path = os.path.join(RESULTS_DIR, "baselines", m, f"splits_{p}.jsonl")
            done = cm.load_done(path)
            for s in SPLITS[p]:
                if s["name"] in done:
                    continue
                r = zc.run_threshold_split(s, STAT[m], y, m, STAT_SHIFT[m])
                r["antenna"] = zc.MAIN_ANTENNA
                cm.append_result(path, r)
            rows = list(cm.load_done(path).values())
            print(f"{m:10s} {p:12s} accuracy {np.mean([r['accuracy'] for r in rows]) * 100:6.2f} %   "
                  f"B {np.mean([r['test_shift']['frame']['accuracy'] for r in rows]) * 100:6.2f} %")

## Step 5: Same-environment test

Every cluster is split in time 70/15/15, with training on the first 70% and testing on the last 15%.

In [ ]:
SAME_ENV_PATH = os.path.join(RESULTS_DIR, "same_environment.json")
if RUN_SAME_ENV:
    se = json.load(open(SAME_ENV_PATH)) if os.path.exists(SAME_ENV_PATH) else {}
    X_ours = None
    for scheme in ("chrono", "random"):
        parts, pidx = zc.same_env_parts(A, POS, scheme, seed=cm.SEED)
        key = f"{scheme}/zheng_svm"
        if key not in se:
            se[key] = zc.same_env_svm(parts, y, cache_size=CACHE_SIZE, max_iter=MAX_ITER)
            json.dump(se, open(SAME_ENV_PATH, "w"), indent=1)
        print(f"{key:24s} accuracy {se[key]['frame']['accuracy'] * 100:6.2f} %")
        for m in METHODS:
            key = f"{scheme}/{m}"
            if key not in se:
                se[key] = zc.same_env_threshold(parts, y, m)
                json.dump(se, open(SAME_ENV_PATH, "w"), indent=1)
            print(f"{key:24s} accuracy {se[key]['frame']['accuracy'] * 100:6.2f} %")
        key = f"{scheme}/our_svm"
        if key not in se:
            if X_ours is None:
                X_ours, y_ours = cm.load_frame_features(cm.ORIG_DATA_DIR)
                assert all(len(X_ours[c]) == len(A[c]) and y_ours[c] == y[c] for c in cm.ALL_CLUSTERS)
            se[key] = zc.same_env_our_svm(X_ours, y, pidx)
            json.dump(se, open(SAME_ENV_PATH, "w"), indent=1)
        print(f"{key:24s} accuracy {se[key]['frame']['accuracy'] * 100:6.2f} %")
    del X_ours

## Step 6: Other antennas (optional)

LOPO splits on four more antennas of the array.

In [ ]:
if RUN_SENSITIVITY:
    for a, where in zc.SENS_ANTENNAS.items():
        Fa, _ = zc.filter_clusters(A, ants.index(a))
        out = os.path.join(RESULTS_DIR, "sensitivity", f"ant{a}")
        path = os.path.join(out, "svm", "splits_lopo.jsonl")
        run_svm_all("lopo", zc.svm_features(Fa), None, os.path.join(out, "svm"), SPLITS["lopo"], table3=False,
                    antenna=a, tag=f"antenna {a} ({where}) ")
        rows = list(cm.load_done(path).values())
        line = f"antenna {a:2d} ({where:6s}) SVM {np.mean([r['accuracy'] for r in rows]) * 100:6.2f} %"
        for m in METHODS:
            pm = os.path.join(out, m, "splits_lopo.jsonl")
            dm = cm.load_done(pm)
            Sa = {c: zc.frame_statistic(Fa[c], m) for c in cm.ALL_CLUSTERS}
            for s in SPLITS["lopo"]:
                if s["name"] not in dm:
                    r = zc.run_threshold_split(s, Sa, y, m)
                    r["antenna"] = a
                    cm.append_result(pm, r)
            line += f"   {m} {np.mean([r['accuracy'] for r in cm.load_done(pm).values()]) * 100:6.2f} %"
        print(line)

## Step 7: Summary

In [ ]:
def mean_of(path, key=lambda r: r["accuracy"]):
    rows = list(cm.load_done(path).values())
    return (float(np.mean([key(r) for r in rows])) * 100, len(rows)) if rows else (float("nan"), 0)

summary = {"unstated_settings": zc.UNSTATED, "svm": {}, "baselines": {}, "same_environment": None}
print(f"{'method':22s} {'protocol':12s} {'n':>3s} {'frame acc':>9s} {'window vote':>11s} {'B: SNR shift':>12s} {'B: LOS rec.':>11s}")
for p in PROTOCOLS:
    path = os.path.join(RESULTS_DIR, "svm", f"splits_{p}.jsonl")
    acc, n = mean_of(path)
    if n:
        wv, _ = mean_of(path, lambda r: r["window_vote"]["accuracy"])
        b, _ = mean_of(path, lambda r: r["test_shift"]["frame"]["accuracy"])
        bl, _ = mean_of(path, lambda r: r["test_shift"]["frame"]["recall_los"])
        summary["svm"][p] = {"accuracy": acc, "window_vote": wv, "test_shift": b, "test_shift_recall_los": bl, "n": n}
        print(f"{'Zheng SVM (RBF)':22s} {p:12s} {n:3d} {acc:9.2f} {wv:11.2f} {b:12.2f} {bl:11.2f}")
for m in METHODS:
    for p in PROTOCOLS:
        path = os.path.join(RESULTS_DIR, "baselines", m, f"splits_{p}.jsonl")
        acc, n = mean_of(path)
        if n:
            b, _ = mean_of(path, lambda r: r["test_shift"]["frame"]["accuracy"])
            summary["baselines"][f"{m}/{p}"] = {"accuracy": acc, "test_shift": b, "n": n}
            print(f"{m:22s} {p:12s} {n:3d} {acc:9.2f} {'':11s} {b:12.2f}")
if os.path.exists(SAME_ENV_PATH):
    se = json.load(open(SAME_ENV_PATH))
    summary["same_environment"] = {k: v["frame"]["accuracy"] * 100 for k, v in se.items()}
    print("\nsame environment (frame accuracy, %):")
    for k, v in summary["same_environment"].items():
        print(f"  {k:24s} {v:6.2f}")
with open(os.path.join(RESULTS_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=1)
shutil.make_archive(os.path.join(os.path.dirname(RESULTS_DIR), "zheng_results"), "zip", RESULTS_DIR)
print("\nwritten:", os.path.join(RESULTS_DIR, "summary.json"), "and zheng_results.zip")